# Chapter 35: Baselines, Metrics, Imbalance, and Error Costs

A synthetic NRG late-shipment example connects confusion counts, thresholds, workload, and expected cost.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parents[1] / 'src'))
from datasciencebook.classification_metrics import classification_summary, threshold_table
print('Imports ready.')


Imports ready.


In [ ]:
rng = np.random.default_rng(35)
n = 1000
actual = np.zeros(n, dtype=int)
actual[rng.choice(n, 80, replace=False)] = 1
scores = np.clip(rng.beta(2, 8, n) + actual * 0.34, 0, 1)
print(f'Late prevalence: {actual.mean():.1%}')


Late prevalence: 8.0%


In [ ]:
baseline = classification_summary(actual, np.zeros(n, dtype=int))
print(f"All-negative accuracy: {baseline['accuracy']:.1%}")
print(f"All-negative recall: {baseline['recall']:.1%}")


All-negative accuracy: 92.0%
All-negative recall: 0.0%


In [ ]:
thresholds = np.round(np.arange(0.10, 0.81, 0.05), 2)
rows = threshold_table(actual, scores, thresholds, false_positive_cost=40, false_negative_cost=500)
feasible = [r for r in rows if r['alerts'] <= 100 and r['recall'] >= 0.60]
best = min(feasible, key=lambda r: r['cost'])
report = {k: round(float(best[k]), 3) if k in ['threshold','precision','recall'] else best[k] for k in ['threshold','alerts','precision','recall','cost']}
print(report)


{'threshold': 0.45, 'alerts': 94, 'precision': 0.649, 'recall': 0.762, 'cost': 10820}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot([r['threshold'] for r in rows], [r['precision'] for r in rows], marker='o', label='Precision')
axes[0].plot([r['threshold'] for r in rows], [r['recall'] for r in rows], marker='o', label='Recall')
axes[0].set(xlabel='Threshold', ylabel='Metric', title='Operating trade-off')
axes[0].legend()
axes[1].plot([r['alerts'] for r in rows], [r['cost'] for r in rows], marker='o')
axes[1].axvline(100, color='firebrick', linestyle='--', label='Capacity')
axes[1].set(xlabel='Alerts', ylabel='Estimated cost', title='Workload and error cost')
axes[1].legend()
fig.tight_layout()
plt.show()


## Interpretation

The majority baseline appears accurate because lateness is rare, but it has zero recall. Lower thresholds find more late shipments while increasing alerts. The selected threshold must satisfy both a recall requirement and the review capacity before cost is compared.


In [ ]:
# Practice: change the false-negative cost and rerun the threshold comparison.
